# Retrieval strategies

Retrieval is an information-retrieval problem inside a RAG system. This notebook compares an inspectable BM25 implementation with a simulated dense ranking and combines them with reciprocal-rank fusion.

## Why multiple signals?

Lexical retrieval is strong for exact identifiers and dense retrieval is often better at paraphrases. Raw scores from different systems are not directly comparable, so rank fusion is a practical baseline.

```mermaid
flowchart LR
 Q[Query] --> L[BM25]
 Q --> D[Dense adapter]
 L --> F[RRF fusion]
 D --> F --> H[Hybrid ranking]
```

In [ ]:
from examples.intermediate.retrieval_strategies import BM25, Document, reciprocal_rank_fusion

documents = [
    Document('auth', 'API keys use the Authorization header and can be rotated.'),
    Document('errors', 'Error E401 means the request is unauthorized.'),
    Document('billing', 'Invoices are available from the billing endpoint.'),
]
bm25 = BM25(documents)
lexical = bm25.search('unauthorized API request', top_k=3)
[(doc.doc_id, round(score, 3)) for doc, score in lexical]

In [ ]:
# A dense retriever would return Documents in this shape.
dense = [documents[1], documents[0], documents[2]]
hybrid = reciprocal_rank_fusion([doc for doc, _ in lexical], dense)
[(doc.doc_id, round(score, 4)) for doc, score in hybrid]

## Exercise

Add a paraphrase query and an exact error-code query. Create a small relevance label for each query, then compare lexical, dense, and hybrid top-k results. Do not choose a winner from one example; turn the examples into a golden evaluation set.